In [1]:
# Set up the proper working directory

import os

current_dir = os.getcwd()
if current_dir.split('/')[-1] == 'notebooks':
    os.chdir('..')

print("Current working directory:", os.getcwd())

Current working directory: /mnt/e/Google Drive/Scolaire/Mines/ue53-NLP/projet/arrayxtract


In [2]:
from scripts.dataset import getPositiveImages, ArreysDetectDataset
from scripts.models import get_model_object_detection
from scripts.utils import *
from scripts.train import train_one_epoch, validate, train

import numpy as np
import os
import matplotlib.pyplot as plt
import json
import gc

from torch import Generator
from torch.utils.data import DataLoader, random_split
from torchvision.models.detection import fasterrcnn_resnet50_fpn, fasterrcnn_resnet50_fpn_v2, fasterrcnn_mobilenet_v3_large_fpn

%load_ext autoreload
%autoreload 2

In [3]:
root = 'data/'

filenames = getPositiveImages(root)

rng = np.random.default_rng(seed=42)
rng.shuffle(filenames)

n_train, n_val = int(0.8 * len(filenames)), int(0.1 * len(filenames))
n_test = len(filenames) - n_train - n_val

train_set = ArreysDetectDataset(root, filenames[:n_train])
val_set = ArreysDetectDataset(root, filenames[n_train:n_train+n_val], train=False)
test_set = ArreysDetectDataset(root, filenames[n_train+n_val:], train=False)

In [4]:
print(f"Train set: {len(train_set)} images")
print(f"Validation set: {len(val_set)} images")
print(f"Test set: {len(test_set)} images")

Train set: 240 images
Validation set: 30 images
Test set: 30 images


In [5]:
train_loader = DataLoader(train_set, batch_size=6, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_set, batch_size=6, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
test_loader = DataLoader(test_set, batch_size=6, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

In [6]:
models = {}

### Possible to train multiple models

#models['fasterrcnn_resnet50_fpn'] = get_model_object_detection(fasterrcnn_resnet50_fpn(weights='DEFAULT', trainable_backbone_layers=2), 2)
models['fasterrcnn_resnet50_fpn_v2'] = get_model_object_detection(fasterrcnn_resnet50_fpn_v2(weights='DEFAULT', trainable_backbone_layers=2), 2)
#models['fasterrcnn_mobilenet_v3_large_fpn'] = get_model_object_detection(fasterrcnn_mobilenet_v3_large_fpn(weights='DEFAULT', trainable_backbone_layers=2), 2)

In [7]:
for model_name, model in models.items():
    print(f'--- {model_name} ---')
    count_all_parameters(model)

--- fasterrcnn_resnet50_fpn_v2 ---
Backbone parameters: 25409536
RPN parameters: 1184015
ROI heads parameters: 15684049
Total trainable parameters: 42277600


In [8]:
from torch.optim import SGD, Adam

In [ ]:
device = get_device()

n_epochs = 10

train_losses, val_losses = {}, {}

for model_name, model in models.items():
    print(f"Training {model_name} custom model...")
    count_all_parameters(model)
    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

    lr_scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,
        gamma=0.1
    )

    train_losses[model_name], val_losses[model_name] = train(model, optimizer, train_loader, val_loader, device=device, lr_scheduler=lr_scheduler, num_epochs=n_epochs)

    gc.collect()
    torch.cuda.empty_cache()

In [11]:
def convert_losses_to_array(losses):
    for model_name, loss_values in losses.items():
        if isinstance(loss_values, np.ndarray):
            losses[model_name] = [list(loss) for loss in loss_values]
        if isinstance(loss_values, dict):
            losses[model_name] = [list(loss) for loss in loss_values.values()]
    return losses

train_losses_copy = train_losses.copy()
train_losses = convert_losses_to_array(train_losses_copy)

val_losses_copy = val_losses.copy()
val_losses = convert_losses_to_array(val_losses_copy)

In [10]:
## Uncomment to save the results

# with open(f"results/{model_name}_sample_train.json", "w") as outfile: 
#     json.dump(train_losses, outfile)

# with open(f"results/{model_name}_sample_val.json", "w") as outfile:
#     json.dump(val_losses, outfile)

In [12]:
# ## Execute to save the model's weights

# model_path = f"models/{model_name}_model.pth"
# torch.save(model.state_dict(), model_path)
# print(f"Model saved to {model_path}")

Model saved to models/fasterrcnn_resnet50_fpn_v2_model.pth
